In [3]:
import pandas as pd

In [4]:
dt = pd.read_excel('desafio_padroes_vendas.xlsx')

In [5]:
dt.groupby('Cidade').head(15).sort_values(by='Valor_Produto', ascending=False)

,Pedido,Cidade,Categoria,Produto,Quantidade,Valor_Produto,Frete,Desconto_%,Forma_Pagamento
7,1008,Boa Vista,Eletrônicos,Notebook,1,4500,120,0,Pix
11,1012,Belém,Eletrônicos,Notebook,2,4300,85,7,Cartão
0,1001,Manaus,Eletrônicos,Notebook,2,4200,80,5,Cartão
9,1010,Porto Velho,Móveis,Armário,2,1800,250,20,Cartão
3,1004,Belém,Eletrônicos,Monitor,5,1100,70,8,Cartão
8,1009,Rio Branco,Eletrônicos,Monitor,7,980,60,12,Boleto
13,1014,Manaus,Móveis,Mesa,4,980,200,10,Boleto
2,1003,Belém,Móveis,Mesa,3,950,180,10,Boleto
10,1011,Manaus,Informática,HD Externo,6,550,35,5,Pix
4,1005,Porto Velho,Informática,SSD,8,420,30,0,Pix


In [12]:
dt.columns

Index(['Pedido', 'Cidade', 'Categoria', 'Produto', 'Quantidade',
       'Valor_Produto', 'Frete', 'Desconto_%', 'Forma_Pagamento',
       'Valor_Bruto', 'Receita_Liquida', 'Receita_Final'],
      dtype='str')

In [7]:
# Criando a nova coluna com o produto das duas variáveis
dt['Valor_Total_Sem_Desconto'] = dt['Valor_Produto'] * dt['Quantidade']

In [8]:
# Renomeando colunas específicas
dt = dt.rename(columns={'Valor_Total_Sem_Desconto': 'Valor_Bruto'})

In [9]:
dt['Receita_Liquida'] = dt['Valor_Bruto'] - (0.01 * dt['Desconto_%'] * dt['Valor_Bruto'])

In [10]:
dt['Receita_Final'] = dt['Receita_Liquida'] - dt['Frete']

In [11]:
df = dt # Eu estava trabalhando com a variável com nome incorreto desde o início haha

### Missão 1: Descobrir padrões

#### 1. Existe alguma cidade que compra produtos mais caros?

In [13]:
df.groupby('Cidade')['Valor_Produto'].mean().sort_values(ascending=False)

Cidade
Boa Vista      2455.000000
Belém          2116.666667
Manaus         1206.000000
Porto Velho    1110.000000
Rio Branco      490.000000
Name: Valor_Produto, dtype: float64

In [14]:
# Criando uma tabela cruzando Produto e Cidade, com a média de preço
tabela_comparativa = df.pivot_table(
    index='Produto',
    columns='Cidade',
    values='Valor_Produto',
    aggfunc='mean'
)

print(tabela_comparativa)

Cidade       Belém  Boa Vista  Manaus  Porto Velho  Rio Branco
Produto                                                       
Armário        NaN        NaN     NaN       1800.0         NaN
Cadeira        NaN        NaN     NaN          NaN       380.0
HD Externo     NaN        NaN   550.0          NaN         NaN
Mesa         950.0        NaN   980.0          NaN         NaN
Monitor     1100.0        NaN     NaN          NaN       980.0
Mouse          NaN        NaN   120.0          NaN       110.0
Notebook    4300.0     4500.0  4200.0          NaN         NaN
SSD            NaN      410.0     NaN        420.0         NaN
Teclado        NaN        NaN   180.0          NaN         NaN


In [15]:
cidades_mais_caras_por_produto = tabela_comparativa.idxmax(axis=1)

print(cidades_mais_caras_por_produto)

Produto
Armário       Porto Velho
Cadeira        Rio Branco
HD Externo         Manaus
Mesa               Manaus
Monitor             Belém
Mouse              Manaus
Notebook        Boa Vista
SSD           Porto Velho
Teclado            Manaus
dtype: str


#### 2. Alguma cidade compra em maior quantidade?

In [17]:
# Quantidade total de itens comprados por cada cidade
df.groupby('Cidade')['Quantidade'].sum().sort_values(ascending=False)


Cidade
Rio Branco     47
Manaus         34
Boa Vista      15
Belém          10
Porto Velho    10
Name: Quantidade, dtype: int64

In [18]:
# Média de itens levados por cada registro de compra
df.groupby('Cidade')['Quantidade'].mean().sort_values(ascending=False)

Cidade
Rio Branco     15.666667
Boa Vista       7.500000
Manaus          6.800000
Porto Velho     5.000000
Belém           3.333333
Name: Quantidade, dtype: float64

#### 3. Existe relação entre valor do frete e quantidade vendida?

In [22]:
# Calculando a correlação entre o valor do frete e a quantidade
df['Frete'].corr(df['Quantidade'])

np.float64(-0.5794551197964231)

In [24]:
df.groupby('Quantidade')['Frete'].mean().sort_index()

# Ou seja, não é necessariamente ligada a quantidade em si, mas sim sobre qual produto está sendo transportado

Quantidade
1     120.000000
2     138.333333
3     180.000000
4     200.000000
5      70.000000
6      35.000000
7      60.000000
8      30.000000
10     25.000000
12     20.000000
14     25.000000
15     90.000000
25     15.000000
Name: Frete, dtype: float64

#### 4. Produtos caros recebem mais desconto?

In [27]:
# A resposta é não. A relação matemática entre o preço e o percentual de desconto é irrelevante
relacao_preco_desconto = df['Valor_Produto'].corr(df['Desconto_%'])

print(f"Correlação Preço x Desconto: {relacao_preco_desconto}")

Correlação Preço x Desconto: 0.004684647301074632


#### 5. Qual categoria gera maior lucro potencial?

In [29]:
df.groupby('Categoria')['Receita_Liquida'].sum().sort_values(ascending=False)
# Com essa execução, podemos concluir que é a de eletrônicos

Categoria
Eletrônicos    35524.8
Informática    14287.0
Móveis         13818.0
Name: Receita_Liquida, dtype: float64

#### 6. Qual forma de pagamento aparece nas maiores compras?

In [38]:
df.nlargest(20, 'Receita_Final')

,Pedido,Cidade,Categoria,Produto,Quantidade,Valor_Produto,Frete,Desconto_%,Forma_Pagamento,Valor_Bruto,Receita_Liquida,Receita_Final
11,1012,Belém,Eletrônicos,Notebook,2,4300,85,7,Cartão,8600,7998.0,7913.0
0,1001,Manaus,Eletrônicos,Notebook,2,4200,80,5,Cartão,8400,7980.0,7900.0
8,1009,Rio Branco,Eletrônicos,Monitor,7,980,60,12,Boleto,6860,6036.8,5976.8
12,1013,Boa Vista,Informática,SSD,14,410,25,0,Pix,5740,5740.0,5715.0
3,1004,Belém,Eletrônicos,Monitor,5,1100,70,8,Cartão,5500,5060.0,4990.0
5,1006,Rio Branco,Móveis,Cadeira,15,380,90,15,Pix,5700,4845.0,4755.0
7,1008,Boa Vista,Eletrônicos,Notebook,1,4500,120,0,Pix,4500,4500.0,4380.0
4,1005,Porto Velho,Informática,SSD,8,420,30,0,Pix,3360,3360.0,3330.0
13,1014,Manaus,Móveis,Mesa,4,980,200,10,Boleto,3920,3528.0,3328.0
10,1011,Manaus,Informática,HD Externo,6,550,35,5,Pix,3300,3135.0,3100.0


In [37]:
# 20 maiores vendas do df
maiores_vendas = df.nlargest(20, 'Receita_Final')

metodos_top_vendas = maiores_vendas['Forma_Pagamento'].value_counts()

print(metodos_top_vendas)

# Logo, podemos verificar que das maiores vendas, 7 delas foram com o método de pix e 5 com cartão

Forma_Pagamento
Pix       7
Cartão    5
Boleto    3
Name: count, dtype: int64


#### 7. Existe algum produto que vende muito, mas gera pouco faturamento?

In [40]:
desempenho_produtos = df.groupby('Produto').agg({
    'Quantidade': 'sum',
    'Receita_Final': 'sum'
})

# Ordena a tabela pelos produtos que mais saíram do estoque (maior quantidade)
produtos_mais_vendidos = desempenho_produtos.sort_values(by='Quantidade', ascending=False)

print(produtos_mais_vendidos)

# Neste caso, podemos verificar que é o 'mouse'

            Quantidade  Receita_Final
Produto                              
Mouse               35         3910.0
SSD                 22         9045.0
Cadeira             15         4755.0
Teclado             12         2032.0
Monitor             12        10966.8
Mesa                 7         5713.0
HD Externo           6         3100.0
Notebook             5        20193.0
Armário              2         2630.0
